# FASE 2 — Control de calidad de los datos

En este notebook analizaremos la calidad de la base histórica
descargada desde MetaTrader 5.

Estudiaremos por separado:

- Las velas M1.
- Los ticks Bid/Ask.

Comprobaremos estructura, valores nulos, duplicados, coherencia OHLC,
huecos temporales y sincronización entre velas y ticks.

Los archivos originales de `data/raw/` no serán modificados.

In [1]:
from pathlib import Path

import pandas as pd


# --------------------------------------------------
# CONFIGURACIÓN
# --------------------------------------------------

DATA_RAW = Path("../data/raw")

SNAPSHOT_DATE = "20260908"

M1_FILE = (
    DATA_RAW
    / f"eurusd_m1_{SNAPSHOT_DATE}.parquet"
)

TICKS_FILE = (
    DATA_RAW
    / f"eurusd_ticks_{SNAPSHOT_DATE}.parquet"
)

# Tamaño de un punto para EURUSD con 5 decimales
POINT = 0.00001


print("Archivo M1:")
print(M1_FILE.resolve())

print("\nArchivo ticks:")
print(TICKS_FILE.resolve())

Archivo M1:
C:\ALL\TECNICO\TRADING\CURSOS\TUTORIAS\PROYECTO2\mt5_microstructure_lab\data\raw\eurusd_m1_20260908.parquet

Archivo ticks:
C:\ALL\TECNICO\TRADING\CURSOS\TUTORIAS\PROYECTO2\mt5_microstructure_lab\data\raw\eurusd_ticks_20260908.parquet


## 1. Comprobar que existen los archivos

Antes de realizar cualquier análisis verificamos que los dos archivos
creados en la Fase 1 existen realmente.

In [2]:
if not M1_FILE.exists():
    raise FileNotFoundError(
        f"No existe el archivo:\n{M1_FILE.resolve()}"
    )

if not TICKS_FILE.exists():
    raise FileNotFoundError(
        f"No existe el archivo:\n{TICKS_FILE.resolve()}"
    )


print("Archivo M1 encontrado correctamente.")
print("Archivo de ticks encontrado correctamente.")

Archivo M1 encontrado correctamente.
Archivo de ticks encontrado correctamente.


## 2. Cargar la base histórica

Cargamos las velas M1 y los ticks desde los archivos Parquet.

Conservamos primero el orden original para poder comprobar
si los datos fueron almacenados cronológicamente.

In [3]:
m1_raw = pd.read_parquet(
    M1_FILE
)

ticks_raw = pd.read_parquet(
    TICKS_FILE
)


m1_raw["time"] = pd.to_datetime(
    m1_raw["time"],
    utc=True
)

ticks_raw["time"] = pd.to_datetime(
    ticks_raw["time"],
    utc=True
)


print(
    "Velas M1 cargadas:",
    len(m1_raw)
)

print(
    "Ticks cargados:",
    len(ticks_raw)
)

Velas M1 cargadas: 99999
Ticks cargados: 1064413


## 3. Examinar la estructura de las dos bases

Revisamos dimensiones, nombres de columnas y tipos de datos antes
de realizar los controles de calidad.

In [4]:
print(
    "========== VELAS M1 =========="
)

print(
    "Dimensión:",
    m1_raw.shape
)

print("\nColumnas:")
print(m1_raw.columns.tolist())

print("\nTipos:")
print(m1_raw.dtypes)


print(
    "\n\n========== TICKS =========="
)

print(
    "Dimensión:",
    ticks_raw.shape
)

print("\nColumnas:")
print(ticks_raw.columns.tolist())

print("\nTipos:")
print(ticks_raw.dtypes)

========== VELAS M1 ==========
Dimensión: (99999, 8)

Columnas:
['time', 'open', 'high', 'low', 'close', 'tick_volume', 'spread', 'real_volume']

Tipos:
time           datetime64[ms, UTC]
open                       float64
high                       float64
low                        float64
close                      float64
tick_volume                 uint64
spread                       int32
real_volume                 uint64
dtype: object


========== TICKS ==========
Dimensión: (1064413, 10)

Columnas:
['time', 'bid', 'ask', 'last', 'volume', 'time_msc', 'flags', 'volume_real', 'spread_price', 'spread_points']

Tipos:
time             datetime64[ms, UTC]
bid                          float64
ask                          float64
last                         float64
volume                        uint64
time_msc                       int64
flags                         uint32
volume_real                  float64
spread_price                 float64
spread_points                float64

## 4. Comprobar el orden cronológico

Antes de ordenar los datos comprobamos si los archivos originales
ya estaban almacenados de forma cronológica.

In [5]:
m1_originalmente_ordenado = (
    m1_raw["time"]
    .is_monotonic_increasing
)

ticks_originalmente_ordenados = (
    ticks_raw["time"]
    .is_monotonic_increasing
)


print(
    "M1 originalmente ordenado:",
    m1_originalmente_ordenado
)

print(
    "Ticks originalmente ordenados:",
    ticks_originalmente_ordenados
)


# --------------------------------------------------
# COPIAS DE TRABAJO
# --------------------------------------------------

m1 = (
    m1_raw
    .sort_values("time")
    .reset_index(drop=True)
    .copy()
)

ticks = (
    ticks_raw
    .sort_values("time")
    .reset_index(drop=True)
    .copy()
)


print(
    "\nM1 ordenado para análisis:",
    m1["time"].is_monotonic_increasing
)

print(
    "Ticks ordenados para análisis:",
    ticks["time"].is_monotonic_increasing
)

M1 originalmente ordenado: True
Ticks originalmente ordenados: True

M1 ordenado para análisis: True
Ticks ordenados para análisis: True


## 5. Analizar valores nulos

Los valores nulos pueden provocar errores posteriores en indicadores,
señales y backtesting.

Contaremos los nulos de cada columna sin modificar todavía ninguna fila.

In [6]:
print(
    "========== NULOS M1 =========="
)

m1_nulls = (
    m1
    .isna()
    .sum()
)

display(
    m1_nulls.to_frame(
        name="nulos"
    )
)

print(
    "\nTotal de nulos M1:",
    m1_nulls.sum()
)


print(
    "\n========== NULOS TICKS =========="
)

ticks_nulls = (
    ticks
    .isna()
    .sum()
)

display(
    ticks_nulls.to_frame(
        name="nulos"
    )
)

print(
    "\nTotal de nulos ticks:",
    ticks_nulls.sum()
)

========== NULOS M1 ==========


,nulos
time,0
open,0
high,0
low,0
close,0
tick_volume,0
spread,0
real_volume,0



Total de nulos M1: 0

========== NULOS TICKS ==========


,nulos
time,0
bid,0
ask,0
last,0
volume,0
time_msc,0
flags,0
volume_real,0
spread_price,0
spread_points,0



Total de nulos ticks: 0


# PARTE A — CONTROL DE LAS VELAS M1

## 6. Analizar duplicados en las velas M1

Comprobamos dos posibles problemas:

1. Dos velas con exactamente la misma fecha y hora.
2. Dos filas completamente idénticas.

In [7]:
m1_duplicate_time = (
    m1["time"]
    .duplicated()
    .sum()
)

m1_exact_duplicates = (
    m1
    .duplicated()
    .sum()
)


print(
    "Fechas M1 duplicadas:",
    m1_duplicate_time
)

print(
    "Filas M1 exactamente duplicadas:",
    m1_exact_duplicates
)

Fechas M1 duplicadas: 0
Filas M1 exactamente duplicadas: 0


## 7. Comprobar la lógica OHLC

Toda vela debe cumplir determinadas relaciones lógicas:

- High debe ser igual o superior a Open, Close y Low.
- Low debe ser igual o inferior a Open, Close y High.
- Los precios deben ser positivos.

Una violación de estas reglas indicaría datos inconsistentes.

In [8]:
high_incorrecto = (
    m1["high"]
    <
    m1[
        [
            "open",
            "close",
            "low"
        ]
    ].max(axis=1)
)

low_incorrecto = (
    m1["low"]
    >
    m1[
        [
            "open",
            "close",
            "high"
        ]
    ].min(axis=1)
)

precios_no_positivos = (
    (
        m1[
            [
                "open",
                "high",
                "low",
                "close"
            ]
        ]
        <= 0
    )
    .any(axis=1)
)


print(
    "Velas con High incorrecto:",
    high_incorrecto.sum()
)

print(
    "Velas con Low incorrecto:",
    low_incorrecto.sum()
)

print(
    "Velas con precios <= 0:",
    precios_no_positivos.sum()
)

Velas con High incorrecto: 0
Velas con Low incorrecto: 0
Velas con precios <= 0: 0


## 8. Comprobar volúmenes y spread de las velas

Comprobamos que no existan volúmenes negativos ni spreads negativos.

Un spread igual a cero no se eliminará en esta fase.

In [9]:
tick_volume_negativo = (
    m1["tick_volume"] < 0
).sum()

real_volume_negativo = (
    m1["real_volume"] < 0
).sum()

spread_m1_negativo = (
    m1["spread"] < 0
).sum()


print(
    "tick_volume negativo:",
    tick_volume_negativo
)

print(
    "real_volume negativo:",
    real_volume_negativo
)

print(
    "spread M1 negativo:",
    spread_m1_negativo
)

tick_volume negativo: 0
real_volume negativo: 0
spread M1 negativo: 0


# PARTE B — HUECOS TEMPORALES M1

## 9. Estudiar huecos temporales en M1

Calculamos la diferencia temporal entre cada vela y la anterior.

En un mercado Forex no debemos interpretar automáticamente todos los
huecos como errores, porque existen cierres de mercado, fines de semana
y otros períodos sin cotización.

Aquí únicamente los identificaremos.

In [10]:
m1["time_diff"] = (
    m1["time"]
    .diff()
)

EXPECTED_M1 = pd.Timedelta(
    minutes=1
)

gap_mask = (
    m1["time_diff"]
    > EXPECTED_M1
)


print(
    "Número de saltos superiores a 1 minuto:",
    gap_mask.sum()
)


gaps = pd.DataFrame(
    {
        "desde": (
            m1["time"]
            .shift(1)[gap_mask]
        ),

        "hasta": (
            m1["time"][gap_mask]
        ),

        "duracion": (
            m1["time_diff"][gap_mask]
        )
    }
)


gaps = (
    gaps
    .sort_values(
        "duracion",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    gaps.head(20)
)

Número de saltos superiores a 1 minuto: 16


,desde,hasta,duracion
0,2026-06-05 23:59:00+00:00,2026-06-08 00:00:00+00:00,2 days 00:01:00
1,2026-06-12 23:59:00+00:00,2026-06-15 00:00:00+00:00,2 days 00:01:00
2,2026-06-19 23:59:00+00:00,2026-06-22 00:00:00+00:00,2 days 00:01:00
3,2026-06-26 23:59:00+00:00,2026-06-29 00:00:00+00:00,2 days 00:01:00
4,2026-07-03 23:59:00+00:00,2026-07-06 00:00:00+00:00,2 days 00:01:00
5,2026-07-10 23:59:00+00:00,2026-07-13 00:00:00+00:00,2 days 00:01:00
6,2026-07-17 23:59:00+00:00,2026-07-20 00:00:00+00:00,2 days 00:01:00
7,2026-07-24 23:59:00+00:00,2026-07-27 00:00:00+00:00,2 days 00:01:00
8,2026-07-31 23:59:00+00:00,2026-08-03 00:00:00+00:00,2 days 00:01:00
9,2026-08-07 23:59:00+00:00,2026-08-10 00:00:00+00:00,2 days 00:01:00


# PARTE C — CONTROL DE LOS TICKS

## 10. Comprobar Bid, Ask y spread

Los ticks deben cumplir:

- Bid > 0
- Ask > 0
- Ask >= Bid
- Spread >= 0

También verificaremos que el spread almacenado coincide con Ask - Bid.

In [11]:
bid_no_positivo = (
    ticks["bid"] <= 0
).sum()

ask_no_positivo = (
    ticks["ask"] <= 0
).sum()

ask_menor_bid = (
    ticks["ask"]
    <
    ticks["bid"]
).sum()

spread_negativo = (
    ticks["spread_price"]
    < 0
).sum()


print(
    "Bid <= 0:",
    bid_no_positivo
)

print(
    "Ask <= 0:",
    ask_no_positivo
)

print(
    "Ask < Bid:",
    ask_menor_bid
)

print(
    "Spread negativo:",
    spread_negativo
)


spread_calculado = (
    ticks["ask"]
    - ticks["bid"]
)

spread_error = (
    ticks["spread_price"]
    - spread_calculado
).abs()


print(
    "\nMáxima diferencia entre "
    "spread guardado y Ask-Bid:"
)

print(
    spread_error.max()
)

Bid <= 0: 0
Ask <= 0: 0
Ask < Bid: 0
Spread negativo: 0

Máxima diferencia entre spread guardado y Ask-Bid:
0.0


## 11. Comprobar el spread expresado en puntos

EUR/USD tiene un tamaño de punto de 0.00001 en nuestra cuenta.

Comprobamos que:

spread_points = spread_price / POINT

In [12]:
spread_points_calculado = (
    ticks["spread_price"]
    / POINT
)

spread_points_error = (
    ticks["spread_points"]
    - spread_points_calculado
).abs()


print(
    "Máximo error en spread_points:",
    spread_points_error.max()
)

Máximo error en spread_points: 0.0


# PARTE D — MARCAS TEMPORALES REPETIDAS

## 12. Analizar marcas temporales repetidas en los ticks

En la Fase 1 observamos miles de registros con la misma marca temporal.

Una marca temporal repetida no significa necesariamente que las filas
sean duplicadas.

Pueden existir varios cambios de cotización dentro de la misma
resolución temporal.

Por ello distinguiremos entre:

- Misma marca temporal.
- Fila completamente duplicada.

In [13]:
same_timestamp_count = (
    ticks["time"]
    .duplicated()
    .sum()
)

exact_tick_duplicates = (
    ticks
    .duplicated()
    .sum()
)


print(
    "Registros con marca temporal repetida:",
    same_timestamp_count
)

print(
    "Filas completamente duplicadas:",
    exact_tick_duplicates
)


timestamp_counts = (
    ticks
    .groupby("time")
    .size()
)


timestamps_repetidos = (
    timestamp_counts[
        timestamp_counts > 1
    ]
)


print(
    "\nNúmero de timestamps que aparecen "
    "más de una vez:",
    len(timestamps_repetidos)
)


if len(timestamps_repetidos) > 0:
    print(
        "Máximo número de registros "
        "en un mismo timestamp:",
        timestamps_repetidos.max()
    )

Registros con marca temporal repetida: 3511
Filas completamente duplicadas: 256

Número de timestamps que aparecen más de una vez: 3184
Máximo número de registros en un mismo timestamp: 10


## 13. Examinar ejemplos de timestamps repetidos

Mostramos algunos registros para comprobar si las filas con la misma
marca temporal contienen realmente información diferente.

In [14]:
ticks_same_time = (
    ticks[
        ticks["time"]
        .duplicated(
            keep=False
        )
    ]
)


display(
    ticks_same_time.head(30)
)

,time,bid,ask,last,volume,time_msc,flags,volume_real,spread_price,spread_points
91,2026-09-01 00:27:57.485000+00:00,1.16174,1.16180,0.0,0,1788222477485,1154,0.0,0.00006,6.0
92,2026-09-01 00:27:57.485000+00:00,1.16175,1.16180,0.0,0,1788222477485,1154,0.0,0.00005,5.0
477,2026-09-01 01:00:21.061000+00:00,1.16182,1.16183,0.0,0,1788224421061,1028,0.0,0.00001,1.0
478,2026-09-01 01:00:21.061000+00:00,1.16182,1.16182,0.0,0,1788224421061,1028,0.0,0.00000,0.0
1383,2026-09-01 01:28:47.100000+00:00,1.16167,1.16168,0.0,0,1788226127100,1158,0.0,0.00001,1.0
1384,2026-09-01 01:28:47.100000+00:00,1.16167,1.16167,0.0,0,1788226127100,1028,0.0,0.00000,0.0
3816,2026-09-01 02:30:12.778000+00:00,1.16157,1.16158,0.0,0,1788229812778,1158,0.0,0.00001,1.0
3817,2026-09-01 02:30:12.778000+00:00,1.16158,1.16158,0.0,0,1788229812778,1154,0.0,0.00000,0.0
4409,2026-09-01 02:45:03.210000+00:00,1.16168,1.16168,0.0,0,1788230703210,1154,0.0,0.00000,0.0
4410,2026-09-01 02:45:03.210000+00:00,1.16168,1.16169,0.0,0,1788230703210,1028,0.0,0.00001,1.0


# PARTE E — SINCRONIZACIÓN M1 / TICKS

## 14. Comprobar la cobertura temporal

Las velas M1 cubren un período mucho mayor que los ticks.

Los ticks corresponden solamente a los últimos 7 días.

Comprobaremos que ese período se encuentre incluido dentro del
histórico M1.

In [15]:
m1_start = (
    m1["time"].min()
)

m1_end = (
    m1["time"].max()
)

ticks_start = (
    ticks["time"].min()
)

ticks_end = (
    ticks["time"].max()
)


print(
    "========== M1 =========="
)

print(
    "Desde:",
    m1_start
)

print(
    "Hasta:",
    m1_end
)


print(
    "\n========== TICKS =========="
)

print(
    "Desde:",
    ticks_start
)

print(
    "Hasta:",
    ticks_end
)


ticks_dentro_de_m1 = (
    ticks_start >= m1_start
    and
    ticks_end <= (
        m1_end
        + pd.Timedelta(minutes=1)
    )
)


print(
    "\nTicks dentro del período M1:",
    ticks_dentro_de_m1
)

========== M1 ==========
Desde: 2026-06-02 13:34:00+00:00
Hasta: 2026-09-08 00:16:00+00:00

========== TICKS ==========
Desde: 2026-09-01 00:17:05.574000+00:00
Hasta: 2026-09-08 00:16:56.458000+00:00

Ticks dentro del período M1: True


## 15. Comprobar la correspondencia por minuto

Transformamos temporalmente los ticks a minutos y verificamos que los
minutos donde existen ticks también aparecen en el histórico M1.

Esto no altera los archivos originales.

In [16]:
tick_minutes = pd.Index(
    ticks["time"]
    .dt.floor("min")
    .unique()
)

m1_minutes = pd.Index(
    m1["time"]
    .unique()
)


tick_minutes_without_m1 = (
    tick_minutes
    .difference(
        m1_minutes
    )
)


print(
    "Minutos con ticks pero sin vela M1:",
    len(tick_minutes_without_m1)
)


if len(tick_minutes_without_m1) > 0:
    display(
        pd.DataFrame(
            {
                "minuto_sin_m1":
                tick_minutes_without_m1[:20]
            }
        )
    )

Minutos con ticks pero sin vela M1: 0


## 16. Comprobar el último minuto de la muestra

La última vela cerrada y el último tick deben pertenecer al mismo
minuto de mercado.

In [17]:
last_m1 = (
    m1["time"].max()
)

last_tick = (
    ticks["time"].max()
)

last_tick_minute = (
    last_tick.floor("min")
)


print(
    "Última vela M1:",
    last_m1
)

print(
    "Último tick:",
    last_tick
)

print(
    "Minuto del último tick:",
    last_tick_minute
)


same_last_minute = (
    last_m1
    ==
    last_tick_minute
)


print(
    "\nÚltimo tick pertenece "
    "a la última vela M1:",
    same_last_minute
)

Última vela M1: 2026-09-08 00:16:00+00:00
Último tick: 2026-09-08 00:16:56.458000+00:00
Minuto del último tick: 2026-09-08 00:16:00+00:00

Último tick pertenece a la última vela M1: True


# PARTE F — RESUMEN

## 17. Resumen automático de calidad

Construimos una tabla con los principales controles realizados.

Esta tabla no modifica ninguna observación y sirve únicamente
como diagnóstico de nuestra base histórica.

In [18]:
quality_summary = pd.DataFrame(
    [
        {
            "control": "Nulos M1",
            "valor": int(
                m1_nulls.sum()
            ),
            "estado": (
                "OK"
                if m1_nulls.sum() == 0
                else "REVISAR"
            )
        },

        {
            "control": "Fechas M1 duplicadas",
            "valor": int(
                m1_duplicate_time
            ),
            "estado": (
                "OK"
                if m1_duplicate_time == 0
                else "REVISAR"
            )
        },

        {
            "control": "Duplicados exactos M1",
            "valor": int(
                m1_exact_duplicates
            ),
            "estado": (
                "OK"
                if m1_exact_duplicates == 0
                else "REVISAR"
            )
        },

        {
            "control": "High incorrecto",
            "valor": int(
                high_incorrecto.sum()
            ),
            "estado": (
                "OK"
                if high_incorrecto.sum() == 0
                else "REVISAR"
            )
        },

        {
            "control": "Low incorrecto",
            "valor": int(
                low_incorrecto.sum()
            ),
            "estado": (
                "OK"
                if low_incorrecto.sum() == 0
                else "REVISAR"
            )
        },

        {
            "control": "Precios M1 <= 0",
            "valor": int(
                precios_no_positivos.sum()
            ),
            "estado": (
                "OK"
                if precios_no_positivos.sum() == 0
                else "REVISAR"
            )
        },

        {
            "control": "Nulos ticks",
            "valor": int(
                ticks_nulls.sum()
            ),
            "estado": (
                "OK"
                if ticks_nulls.sum() == 0
                else "REVISAR"
            )
        },

        {
            "control": "Bid <= 0",
            "valor": int(
                bid_no_positivo
            ),
            "estado": (
                "OK"
                if bid_no_positivo == 0
                else "REVISAR"
            )
        },

        {
            "control": "Ask <= 0",
            "valor": int(
                ask_no_positivo
            ),
            "estado": (
                "OK"
                if ask_no_positivo == 0
                else "REVISAR"
            )
        },

        {
            "control": "Ask < Bid",
            "valor": int(
                ask_menor_bid
            ),
            "estado": (
                "OK"
                if ask_menor_bid == 0
                else "REVISAR"
            )
        },

        {
            "control": "Spread negativo",
            "valor": int(
                spread_negativo
            ),
            "estado": (
                "OK"
                if spread_negativo == 0
                else "REVISAR"
            )
        },

        {
            "control": "Duplicados exactos ticks",
            "valor": int(
                exact_tick_duplicates
            ),
            "estado": (
                "OK"
                if exact_tick_duplicates == 0
                else "REVISAR"
            )
        },

        {
            "control": "Ticks fuera de minutos M1",
            "valor": int(
                len(
                    tick_minutes_without_m1
                )
            ),
            "estado": (
                "OK"
                if len(
                    tick_minutes_without_m1
                ) == 0
                else "REVISAR"
            )
        },

        {
            "control": "Último minuto sincronizado",
            "valor": bool(
                same_last_minute
            ),
            "estado": (
                "OK"
                if same_last_minute
                else "REVISAR"
            )
        }
    ]
)


display(
    quality_summary
)

,control,valor,estado
0,Nulos M1,0,OK
1,Fechas M1 duplicadas,0,OK
2,Duplicados exactos M1,0,OK
3,High incorrecto,0,OK
4,Low incorrecto,0,OK
5,Precios M1 <= 0,0,OK
6,Nulos ticks,0,OK
7,Bid <= 0,0,OK
8,Ask <= 0,0,OK
9,Ask < Bid,0,OK


# Conclusión de la Fase 2

En esta fase hemos analizado la calidad de las velas M1 y de los
ticks Bid/Ask sin modificar los archivos originales.

Los huecos temporales y las marcas temporales repetidas no se eliminan
automáticamente. Primero deben interpretarse en el contexto del mercado
Forex y de la estructura de los ticks.

Una vez validada esta fase podremos continuar con el análisis de
microestructura:

`03_bid_ask_spread.ipynb`